In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text('folderpath','')

In [0]:
folderpath = dbutils.widgets.get('folderpath')

In [0]:
df_users = spark.readStream.format('cloudFiles')\
                .option('cloudFiles.format', 'parquet')\
                .option('cloudFiles.schemaLocation', f'abfss://silver@spotifyprojadls.dfs.core.windows.net/{folderpath}/checkpoint')\
                .option('cloudFiles.schemaEvolutionMode', 'addNewColumns')\
                .load(f'abfss://bronze@spotifyprojadls.dfs.core.windows.net/{folderpath}')

In [0]:
df_users.writeStream\
    .format('delta')\
    .option('checkpointLocation',f'abfss://silver@spotifyprojadls.dfs.core.windows.net/{folderpath}/checkpoint')\
    .trigger(once=True)\
    .start(f'abfss://silver@spotifyprojadls.dfs.core.windows.net/{folderpath}/data')
    